# Virussen simuleren

In dit notebook bouwen we stap voor stap een eenvoudige virussimulatie. Voer de cellen van boven naar beneden uit. De grafieken aan het einde laten zien hoe toeval de uitkomst van een simulatie beïnvloedt.

## Voorbereiding

We gebruiken `random` voor de toevalsgebeurtenissen en `matplotlib` voor de grafieken.

In [ ]:
import random
import matplotlib.pyplot as plt

## Stap 1: een virusgenoom maken

Een virusgenoom is een string met willekeurige nucleotiden: `A`, `C`, `G` en `T`. De parameter `length` bepaalt hoeveel letters het genoom krijgt.

In [ ]:
def generate_virus(length: int) -> str:
    return "".join(random.choices("ACGT", k=length))

In [ ]:
virus = generate_virus(4)
print(virus)
assert len(virus) == 4
assert all(nucleotide in "ACGT" for nucleotide in virus)

## Stap 2: muteren

Een mutatie verandert precies één nucleotide. We kiezen een positie en vervangen de letter door een van de andere drie nucleotiden. Bij een lege string is er niets om te muteren, dus geven we die onveranderd terug.

In [ ]:
def mutate(virus: str) -> str:
    if not virus:
        return virus

    index = random.randrange(len(virus))
    alternatives = "".join(
        nucleotide for nucleotide in "ACGT" if nucleotide != virus[index]
    )
    return virus[:index] + random.choice(alternatives) + virus[index + 1:]

In [ ]:
virus = "ACGT"
mutated_virus = mutate(virus)
print(virus, "->", mutated_virus)
assert len(mutated_virus) == len(virus)
assert mutated_virus != virus
assert mutate("") == ""

## Stap 3: virussen laten afsterven

Elk virus krijgt onafhankelijk een kans om af te sterven. De functie maakt een nieuwe lijst met de overlevenden. Een sterftekans van `0` laat iedereen overleven; bij `1` sterft iedereen.

In [ ]:
def kill(viruses: list[str], mortality_prob: float) -> list[str]:
    return [virus for virus in viruses if random.random() >= mortality_prob]

In [ ]:
viruses = ["AAAA", "CCCC", "GGGG"]
assert kill(viruses, 0) == viruses
assert kill(viruses, 1) == []
assert viruses == ["AAAA", "CCCC", "GGGG"]  # De invoerlijst blijft hetzelfde.

## Stap 4: reproduceren

De ouders blijven in de populatie. Elk oudervirus krijgt een reproductiekans; een nieuw kind krijgt vervolgens een kans op mutatie. Kinderen reproduceren pas in een volgende tijdstap.

In [ ]:
def reproduce(
    viruses: list[str], mutation_prob: float, reproduction_prob: float
) -> list[str]:
    population = viruses.copy()
    for virus in viruses:
        if random.random() < reproduction_prob:
            child = mutate(virus) if random.random() < mutation_prob else virus
            population.append(child)
    return population

In [ ]:
viruses = ["AAAA", "CCCC"]
assert reproduce(viruses, mutation_prob=0, reproduction_prob=0) == viruses
assert len(reproduce(viruses, mutation_prob=0, reproduction_prob=1)) == 4
assert viruses == ["AAAA", "CCCC"]  # De invoerlijst blijft hetzelfde.

## Stap 5: resistentie herkennen

In deze simulatie is een virus resistent als de reeks `AAA` direct achter elkaar in het genoom voorkomt.

In [ ]:
def is_resistant(virus: str) -> bool:
    return "AAA" in virus

In [ ]:
assert is_resistant("CAAAT")
assert not is_resistant("AATA")

## Stap 6: de reproductiekans berekenen

Hoe dichter de populatie bij `max_population` komt, hoe kleiner de reproductiekans wordt. Bij een maximale populatie van `0` is de kans `0`, zodat we niet door nul delen.

In [ ]:
def reproduction_probability(
    viruses: list[str], max_reproduction_prob: float, max_population: int
) -> float:
    return (
        (1 - (len(viruses) / max_population)) * max_reproduction_prob
        if max_population > 0
        else 0
    )

In [ ]:
viruses = ["AAAA", "CCCC"]
assert reproduction_probability(viruses, 0.5, 4) == 0.25
assert reproduction_probability(viruses, 0.5, 0) == 0

## Stap 7: de simulatie uitvoeren

Elke tijdstap sterven eerst virussen. Daarna berekenen we de reproductiekans en laten we virussen reproduceren. Vanaf tijdstap 100 kunnen alleen resistente virussen zich voortplanten; niet-resistente virussen blijven wel in de populatie. De lijst met groottes bevat ook de beginsituatie.

In [ ]:
def simulate(
    viruses: list[str],
    mortality_prob: float,
    mutation_prob: float,
    max_reproduction_prob: float,
    max_population: int,
    timesteps: int = 500,
) -> list[int]:
    viruses = viruses.copy()
    population_sizes = [len(viruses)]

    for timestep in range(timesteps):
        viruses = kill(viruses, mortality_prob)
        reproduction_prob = reproduction_probability(
            viruses, max_reproduction_prob, max_population
        )

        if timestep >= 100:
            resistant = [virus for virus in viruses if is_resistant(virus)]
            non_resistant = [virus for virus in viruses if not is_resistant(virus)]
            viruses = non_resistant + reproduce(
                resistant, mutation_prob, reproduction_prob
            )
        else:
            viruses = reproduce(viruses, mutation_prob, reproduction_prob)

        population_sizes.append(len(viruses))

    return population_sizes

In [ ]:
viruses = [generate_virus(4) for _ in range(100)]
result = simulate(viruses, 0.1, 0.1, 0.5, 100, timesteps=10)
assert len(result) == 11  # De beginsituatie plus 10 tijdstappen.
assert result[0] == 100
print(result)

## Grafiek van één simulatie

Elke run is anders door de willekeurige sterfte, reproductie en mutaties.

In [ ]:
viruses = [generate_virus(4) for _ in range(100)]
result = simulate(viruses, 0.1, 0.1, 0.5, 100, timesteps=500)

fig, ax = plt.subplots(figsize=(12, 7))
ax.plot(range(len(result)), result)
ax.set(title="Eén virussimulatie", xlabel="Tijdstap", ylabel="Aantal virussen")
ax.set_ylim(0, 110)
ax.set_xlim(0, 500)
plt.show()

## Grafiek van meerdere simulaties

Door meerdere runs te tekenen, zie je beter hoe verschillend de uitkomsten kunnen zijn.

In [ ]:
n_simulations = 20
fig, ax = plt.subplots(figsize=(12, 7))

for _ in range(n_simulations):
    viruses = [generate_virus(4) for _ in range(100)]
    result = simulate(viruses, 0.1, 0.1, 0.5, 100, timesteps=500)
    ax.plot(range(len(result)), result)

ax.set(title="Meerdere virussimulaties", xlabel="Tijdstap", ylabel="Aantal virussen")
ax.set_ylim(0, 110)
ax.set_xlim(0, 500)
plt.show()

## Genezen of niet genezen?

We tellen hoeveel simulaties eindigen met nul virussen. Dat noemen we in dit model genezen. De taartgrafiek toont de verhouding van genezen en niet-genezen runs.

In [ ]:
n_simulations = 100
n_simulations_cured = 0

for _ in range(n_simulations):
    viruses = [generate_virus(4) for _ in range(100)]
    result = simulate(viruses, 0.1, 0.1, 0.5, 100, timesteps=500)

    if result[-1] == 0:
        n_simulations_cured += 1

labels = ["Genezen", "Niet genezen"]
sizes = [n_simulations_cured, n_simulations - n_simulations_cured]

fig, ax = plt.subplots(figsize=(8, 8))
ax.pie(sizes, labels=labels, autopct="%1i%%", startangle=90)
ax.axis("equal")
ax.set_title("Genezen en niet genezen simulaties")
plt.show()

print(f"Genezen in {n_simulations_cured} van de {n_simulations} simulaties.")